# Enriched Instruments Dataset

Instruments with sector and market cap populated from yfinance (bulk enrichment cascade).

**Columns:** ticker, name, sector, industry, market_cap, business_summary, data_available

## Setup

In [8]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.data.database import get_session
from src.data.models import Instrument

## Load Enriched Instruments

In [9]:
session = get_session()
rows = (
    session.query(
        Instrument.ticker,
        Instrument.name,
        Instrument.sector,
        Instrument.industry,
        Instrument.market_cap,
        Instrument.business_summary,
        Instrument.data_available,
    )
    .filter(
        Instrument.ticker.like("%_US_EQ"),
        Instrument.sector.isnot(None),
        Instrument.sector != "Unknown",
        Instrument.market_cap > 0,
    )
    .order_by(Instrument.sector, Instrument.ticker)
    .all()
)
session.close()

df = pd.DataFrame(
    rows,
    columns=["ticker", "name", "sector", "industry", "market_cap", "business_summary", "data_available"],
)
print(f"Enriched: {len(df):,} instruments")
print(f"  Industry populated: {df['industry'].notna().sum():,} ({100*df['industry'].notna().mean():.1f}%)")
print(f"  Business summary populated: {df['business_summary'].notna().sum():,} ({100*df['business_summary'].notna().mean():.1f}%)")

Enriched: 5,498 instruments


## Backfill Status

Run `poetry run python scripts/backfill_industry_summary.py` to fill industry/summary for instruments that have sector+market_cap but are missing them. Cell below shows how many still need backfill.

In [ ]:
from sqlalchemy import or_
s = get_session()
need = s.query(Instrument).filter(
    Instrument.ticker.like("%_US_EQ"),
    Instrument.sector.isnot(None), Instrument.sector != "Unknown",
    Instrument.market_cap.isnot(None), Instrument.market_cap > 0,
    Instrument.data_available != False,
).filter(
    or_(
        Instrument.industry.is_(None), Instrument.industry == "",
        Instrument.business_summary.is_(None), Instrument.business_summary == "",
    ),
).count()
s.close()
print(f"Instruments still needing industry/summary backfill: {need:,}")

## Dataset

In [10]:
def fmt_market_cap(x):
    if pd.isna(x) or x is None or x <= 0:
        return ""
    if x >= 1e9:
        return f"{x / 1e9:.2f}B"
    if x >= 1e6:
        return f"{x / 1e6:.1f}M"
    return str(int(x))

df["market_cap_fmt"] = df["market_cap"].apply(fmt_market_cap)
df

,ticker,name,sector,industry,market_cap,data_available,market_cap_fmt
0,AA_US_EQ,Alcoa,Basic Materials,Aluminum,1.677757e+10,True,16.78B
1,ABBRF_US_EQ,AbraSilver Resource,Basic Materials,NaN,1.488798e+09,True,1.49B
2,ABEPF_US_EQ,Vision Lithium,Basic Materials,NaN,5.005061e+06,True,5.0M
3,ABSSF_US_EQ,AirBoss of America,Basic Materials,NaN,1.333027e+08,True,133.3M
4,AEM_US_EQ,Agnico Eagle Mines,Basic Materials,NaN,1.038339e+11,True,103.83B
...,...,...,...,...,...,...,...
5493,WNDW_US_EQ,SolarWindow Technologies,Utilities,NaN,2.631162e+07,True,26.3M
5494,WTRG_US_EQ,Essential Utilities,Utilities,NaN,1.179186e+10,True,11.79B
5495,XEL_US_EQ,Xcel Energy,Utilities,NaN,5.110175e+10,True,51.10B
5496,XNGSY_US_EQ,ENN Energy,Utilities,NaN,9.739452e+09,True,9.74B


## Summary by Sector

In [11]:
df.groupby("sector").agg(
    count=("ticker", "count"),
    total_market_cap_b=("market_cap", lambda x: x.sum() / 1e9),
).sort_values("count", ascending=False)

,count,total_market_cap_b
sector,,
Healthcare,1012,5949.161927
Financial Services,998,12246.720121
Industrials,737,9210.977968
Technology,731,8044.150188
Consumer Cyclical,574,6198.005702
Basic Materials,333,3108.896119
Consumer Defensive,284,3208.977874
Communication Services,250,3455.046835
Real Estate,249,1461.273391


## Progress vs Total

In [12]:
s = get_session()
total = s.query(Instrument).filter(Instrument.ticker.like("%_US_EQ")).count()
s.close()
pct = 100 * len(df) / total if total > 0 else 0
print(f"{len(df):,} / {total:,} US equities enriched ({pct:.1f}%)")

5,498 / 6,948 US equities enriched (79.1%)


In [13]:
def market_cap_bucket(x):
    # type: (float|None) -> str
    if pd.isna(x):
        return "Unknown"
    if x >= 1e10:
        return "Large"
    elif x >= 2e9:
        return "Mid"
    elif x > 0:
        return "Small"
    else:
        return "Unknown"

df["market_cap_bucket"] = df["market_cap"].apply(market_cap_bucket)

# Group by sector and market cap bucket, count tickers
summary = (
    df.groupby(["sector", "market_cap_bucket"])["ticker"]
      .count()
      .reset_index(name="count")
      .pivot(index="sector", columns="market_cap_bucket", values="count")
      .fillna(0)
      .astype(int)
)

# Add "Total" column (per sector)
summary["Total"] = summary.sum(axis=1)
# Add a grand total row
summary.loc["Total"] = summary.sum(axis=0)

summary

market_cap_bucket,Large,Mid,Small,Total
sector,,,,
Basic Materials,73,97,163,333
Communication Services,64,54,132,250
Consumer Cyclical,131,155,288,574
Consumer Defensive,73,66,145,284
Energy,50,62,100,212
Financial Services,195,168,635,998
Healthcare,116,151,745,1012
Industrials,204,180,353,737
Real Estate,42,67,140,249
